In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225

In [ ]:
token = ""


In [5]:
import json
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
MAX_SEQ_LENGTH = 2048   # veri setindeki p95 ~510, maks ~638 token (thinking dahil, yaklaşık ölçüm) + tampon
LORA_RANK = 16
LORA_ALPHA = 16
BATCH_SIZE = 1
GRAD_ACCUM = 8          # etkin batch = 1*8 = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # otomatik seçilsin (bf16 destekleniyorsa o, yoksa fp16)
)

==((====))==  Unsloth 2026.7.5: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_rslora = True,
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

Unsloth 2026.7.5 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [6]:
from huggingface_hub import login
login(token)

In [14]:
from datasets import load_dataset

dataset = load_dataset("uzcaliskan/magibu_dataset_drilling", split="train")

In [15]:
dataset

Dataset({
    features: ['content', 'images', 'role', 'thinking', 'tool_calls'],
    num_rows: 5568
})

In [16]:
def format_chat_data(batch):
    texts = []
    # Verideki tüm elemanları ikişerli (user ve assistant) alıyoruz
    for i in range(0, len(batch["role"]) - 1, 2):
        if batch["role"][i] == "user" and batch["role"][i+1] == "assistant":
            user_content = batch["content"][i]
            asst_content = batch["content"][i+1]

            messages = [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": asst_content}
            ]

            # Tokenizer'ın sohbet şablonunu uyguluyoruz
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)

    return {"text": texts}

# Veri setini dönüştürüyoruz
formatted_dataset = dataset.map(format_chat_data, batched=True, remove_columns=dataset.column_names)

Map:   0%|          | 0/5568 [00:00<?, ? examples/s]

In [17]:
# from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    eval_dataset = None,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,


    args = UnslothTrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,

        # Use warmup_ratio and num_train_epochs for longer runs!
        #max_steps = 10,
        warmup_steps = 2,
        # warmup_ratio = 0.1,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 5e-5,
        embedding_learning_rate = 1e-6,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2784 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [18]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,784 | Num Epochs = 1 | Total steps = 696
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.695773
2,3.058656
3,2.755238
4,2.813383
5,2.581403
6,1.976226
7,2.161438
8,2.299991
9,2.067900
10,2.513099


Unsloth: Restored added_tokens_decoder metadata in /content/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/checkpoint-696/tokenizer_config.json.


In [20]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/kth_tekop_sondaj_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in kth_tekop_sondaj_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [11:49<00:00, 177.31s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)




Unsloth: Merging weights into 16bit:   0%|          | 0/4 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  25%|██▌       | 1/4 [00:57<02:52, 57.34s/it]

Unsloth: Merging weights into 16bit:  50%|█████     | 2/4 [01:57<01:58, 59.17s/it]

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [02:58<00:59, 59.86s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:12<00:00, 48.14s/it]


Unsloth: Merge process complete. Saved to `/content/kth_tekop_sondaj_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10107-mix-1911198 (app-b10107-mix-1911198-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['kth_tekop_sondaj_gguf_gguf/qwen3-8b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed

{'save_directory': 'kth_tekop_sondaj_gguf',
 'gguf_directory': 'kth_tekop_sondaj_gguf_gguf',
 'gguf_files': ['kth_tekop_sondaj_gguf_gguf/qwen3-8b.Q4_K_M.gguf'],
 'modelfile_location': 'kth_tekop_sondaj_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
model.push_to_hub_gguf(
    "uzcaliskan/kth-tekop-sondaj-model",   # var olan model reponun içine push eder
    tokenizer,
    quantization_method = "q4_k_m",
    token = token,     # yazma izinli token
)

In [19]:
model.push_to_hub_merged("uzcaliskan/kth-tekop-sondaj-model",
                         tokenizer,
                         token = token,
                         private=True)

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in uzcaliskan/kth-tekop-sondaj-model-yeni/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   2%|2         | 32.0MB / 1.58GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [10:10<00:00, 152.68s/it]


Unsloth: Merge process complete. Saved to `/content/uzcaliskan/kth-tekop-sondaj-model-yeni`


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="C:/Users/Msi/Desktop/Magibu/kth_tekop_sondaj_q4_k_m.gguf",
    path_in_repo="kth_tekop_sondaj_q4_k_m.gguf",
    repo_id="uzcaliskan/kth-tekop-sondaj-model",  # var olan model reponun içine
    repo_type="model"
    token=token,  # yazma izinli token
)